In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib
import matplotlib as mpl
import math
%matplotlib inline

style_file = pathlib.Path.cwd().parent / "plot_style.mplstyle"
plt.style.use(["default", style_file])

inputpath_data = pathlib.Path.cwd() / 'input_data/calcium_imaging' 
inputpath_data.mkdir(parents=True, exist_ok=True)
savepath_data = pathlib.Path.cwd() / 'output_data/calcium_imaging' 
savepath_data.mkdir(parents=True, exist_ok=True)
savepath_fig = pathlib.Path.cwd() / 'output_fig/calcium_imaging' 
savepath_fig.mkdir(parents=True, exist_ok=True)

%config InlineBackend.figure_format = 'retina'

In [ ]:
def factorize_dataset(df, cols_to_factorize_on, new_col_name):
    df[new_col_name] = df.groupby(cols_to_factorize_on,sort=False).ngroup().astype(int)
    return df

def get_stim_params(x, non_stimulus_specs):
    return [e for e in x if e not in non_stimulus_specs]

def average_and_baseline(df):
    averaged = pd.DataFrame(df.groupby(col_names_except(df, ["trial", "value"]), group_keys=False)['value'].mean()).rename(columns={'value': 'averaged_value'}).reset_index()
    baselined_values = averaged.groupby(col_names_except(averaged, ["time", "averaged_value"]), group_keys=False)[['time','averaged_value']].apply(lambda x: subtract_baseline(x, 'averaged_value'))
    baselined = averaged.merge(baselined_values).rename(columns={'value_baselined':'df/f'})
    return baselined.drop(columns='averaged_value')

def col_names_except(df, remove_names):
    all_names = df.columns.to_list()
    names_reduced = [e for e in all_names if e not in remove_names]
    return names_reduced

def subtract_baseline(df, values_column):
    base = df.query("time < 0.0")[values_column].values.mean()
    df = df.copy()
    df["value_baselined"] = df[values_column] - base
    return df

In [ ]:
dtypes = {'bg_intensity': 'float64',
 'edge_intensity': 'float64',
 'filename_base': 'O',
 'fly': 'O',
 'genotype': 'O',
 'max_intensity': 'float64',
 'min_intensity': 'float64',
 'mode [sq/sin]': 'O',
 'name': 'O',
 'roi': 'O',
 'roi_type': 'O',
 'rotation [deg]': 'float64',
 'start pause [sec]': 'float64',
 'stimulus duration [s]': 'float64',
 'time': 'float64',
 'trial': 'O',
 'value': 'float64',
 'velocity [deg/s]': 'float64',
 'wavelength [deg]': 'float64'}

In [ ]:
data = pd.read_csv(inputpath_data / 'preprocessed_calcium_imaging.csv', dtype=dtypes)
imaging_specs = ["fly", "genotype", "filename_base"]
roi_specs = ["roi", "roi_type"]
stimulus_specs = ["name", "start pause [sec]", "bg_intensity", "edge_intensity", "rotation [deg]", "velocity [deg/s]",
                  "wavelength [deg]", "max_intensity", "min_intensity", "mode [sq/sin]", "stimulus duration [s]",
                  "pulse length [s]", "intensity",
                  "trial", "time"]
non_stimulus_specs = imaging_specs + roi_specs + ['name', 'trial', 'time', 'value', 'df/f', 'genotype_sort']

In [ ]:
cols_to_factorize_on = ["fly", "filename_base", "roi_type", "roi"]
data_factorized = factorize_dataset(data.reset_index(), cols_to_factorize_on, new_col_name="unique_roi")
data_factorized['genotype'] = data_factorized['genotype'].map({'T4T5adGC8m_a': 'T4T5ad', 'HisCl1doubleRNAi_a': 'HisCl1_RNAi', 'RdlRNAi_a': 'Rdl_RNAi', 'OrtRNAi_a': 'Ort_RNAi', 'RdlOrtRNAi_a': 'Rdl_Ort_RNAi'})

lookup_genotype = {'T4T5ad': 0, 'HisCl1_RNAi': 1, 'Rdl_RNAi': 2, 'Ort_RNAi': 3, 'Rdl_Ort_RNAi': 4}

In [ ]:
palette_genotypes = {'T4T5ad': '#000000', 'HisCl1_RNAi': '#828288', 'Rdl_RNAi': '#52c3c3', 'Ort_RNAi':'#e8872e', 'Rdl_Ort_RNAi': '#662864'}
genotype_formatting = {'T4T5ad': 'Control', 'HisCl1_RNAi': 'HisCl1 RNAi', 'Rdl_RNAi': 'Rdl RNAi', 'Ort_RNAi':'ort RNAi', 'Rdl_Ort_RNAi': 'Rdl + ort RNAi'}

In [ ]:
# Sort according to genotype
df = data_factorized.copy()
df_mapping = pd.DataFrame({'genotype': ['T4T5ad', 'HisCl1_RNAi', 'Rdl_RNAi', 'Ort_RNAi', 'Rdl_Ort_RNAi']})

sort_mapping = df_mapping.reset_index().set_index('genotype')
sort_mapping
df['genotype_sort'] = df['genotype'].map(sort_mapping['index'])
df = df.sort_values(['genotype_sort', 'name', 'unique_roi', 'trial', 'time']).reset_index(drop=True)

df = df[col_names_except(df, 'index')]

df['experiment'] = None
df.loc[df['filename_base'].str.contains('dir_tuning'), 'experiment'] = 'dir_tuning'
df.loc[df['filename_base'].str.contains('temp_contrast_tuning'), 'experiment'] = 'temp_tuning'
df.loc[df['filename_base'].str.contains('fieldflicker'), 'experiment'] = 'flashes'
df.loc[df['filename_base'].str.contains('ON_OFF_selectivity'), 'experiment'] = 'ON_OFF_selectivity'

In [ ]:
### Create lists of all stimulus parameters for each stimulus type ###

stimuli = []
column_names = []

for experiment, dataset in df.groupby('experiment'):
    stimuli.append(experiment)
    column_names.append(dataset.dropna(how='any', axis=1).columns.to_list())
    
stimuli_to_loop = pd.DataFrame({'experiment': stimuli, 'column_names': column_names})
stimuli_to_loop['stim_params'] = stimuli_to_loop['column_names'].apply(lambda row: get_stim_params(row, non_stimulus_specs))

In [ ]:
### Make separate dataframes per stimulus type, average over trials and baseline subtraction ###

dir_tuning_index = stimuli_to_loop['experiment'].loc[lambda x: x=='dir_tuning'].index[0]
dir_tuning_name = stimuli_to_loop['experiment'][dir_tuning_index]
dir_tuning_data = df.query("experiment == @dir_tuning_name")[stimuli_to_loop.loc[dir_tuning_index, 'column_names']]
dir_tuning_data = average_and_baseline(dir_tuning_data)
dir_tuning_data["contrast"] = np.around(abs(dir_tuning_data["edge_intensity"] - dir_tuning_data["bg_intensity"]) / 255, decimals=2)
dir_tuning_data["contrast"] = dir_tuning_data["contrast"].replace(0.49, 0.5) # calculated 50% contrast rounds to 49% but still want to display it at 50%
dir_tuning_data["polarity"] = np.sign(dir_tuning_data["edge_intensity"] - dir_tuning_data["bg_intensity"])
dir_tuning_data = dir_tuning_data.drop(columns=["edge_intensity", "bg_intensity"])
dir_tuning_data = dir_tuning_data.sort_values(['genotype_sort', 'unique_roi', 'time']).reset_index(drop=True)

temp_tuning_index = stimuli_to_loop['experiment'].loc[lambda x: x=='temp_tuning'].index[0]
temp_tuning_name = stimuli_to_loop['experiment'][temp_tuning_index]
temp_tuning_data = df.query("experiment == @temp_tuning_name")[stimuli_to_loop.loc[temp_tuning_index, 'column_names']]
temp_tuning_data = average_and_baseline(temp_tuning_data)
temp_tuning_data["contrast"] = np.around(abs(temp_tuning_data["max_intensity"] - temp_tuning_data["min_intensity"]) / 255, decimals=2)
temp_tuning_data["contrast"] = temp_tuning_data["contrast"].replace(0.49, 0.5) # calculated 50% contrast rounds to 49% but still want to display it at 50%
temp_tuning_data["frequency"] = temp_tuning_data["velocity [deg/s]"] / temp_tuning_data["wavelength [deg]"]
direction_mapping = {342: 'PD', 0: 'PD', 18: 'PD', 36: 'PD', 162: 'ND', 180: 'ND', 198: 'ND', 216: 'ND'}
temp_tuning_data['direction'] = temp_tuning_data['rotation [deg]'].map(direction_mapping)
temp_tuning_data = temp_tuning_data.drop(columns=["max_intensity", "min_intensity", 'rotation [deg]', 'start pause [sec]', 'velocity [deg/s]', 'mode [sq/sin]',	'stimulus duration [s]', 'wavelength [deg]'])
temp_tuning_data = temp_tuning_data.sort_values(['genotype_sort', 'unique_roi', 'time']).reset_index(drop=True)

flashes_index = stimuli_to_loop['experiment'].loc[lambda x: x=='flashes'].index[0]
flashes_name = stimuli_to_loop['experiment'][flashes_index]
flashes_data = df.query("experiment == @flashes_name")[stimuli_to_loop.loc[flashes_index, 'column_names']]
flashes_data = average_and_baseline(flashes_data)
flashes_data["contrast"] = np.around(abs(flashes_data["max_intensity"] - flashes_data["min_intensity"]) / 255, decimals=2)
flashes_data["contrast"] = flashes_data["contrast"].replace(0.49, 0.5) # calculated 50% contrast rounds to 49% but still want to display it at 50%
flashes_data["polarity"] = np.sign(flashes_data["max_intensity"] - flashes_data["min_intensity"])
flashes_data = flashes_data.drop(columns=["max_intensity", "min_intensity"])
flashes_data = flashes_data.sort_values(['genotype_sort', 'unique_roi', 'time']).reset_index(drop=True)

ON_OFF_selectivity_index = stimuli_to_loop['experiment'].loc[lambda x: x=='ON_OFF_selectivity'].index[0]
ON_OFF_selectivity_name = stimuli_to_loop['experiment'][ON_OFF_selectivity_index]
ON_OFF_selectivity_data = df.query("experiment == @ON_OFF_selectivity_name")[stimuli_to_loop.loc[ON_OFF_selectivity_index, 'column_names']]
ON_OFF_selectivity_data = average_and_baseline(ON_OFF_selectivity_data)
ON_OFF_selectivity_data["contrast"] = np.around(abs(ON_OFF_selectivity_data["edge_intensity"] - ON_OFF_selectivity_data["bg_intensity"]) / 255, decimals=2)
ON_OFF_selectivity_data["contrast"] = ON_OFF_selectivity_data["contrast"].replace(0.49, 0.5) # calculated 50% contrast rounds to 49% but still want to display it at 50%
ON_OFF_selectivity_data["polarity"] = np.sign(ON_OFF_selectivity_data["edge_intensity"] - ON_OFF_selectivity_data["bg_intensity"])
direction_mapping = {342: 'PD', 0: 'PD', 18: 'PD', 36: 'PD', 162: 'ND', 180: 'ND', 198: 'ND', 216: 'ND'}
ON_OFF_selectivity_data['direction'] = ON_OFF_selectivity_data['rotation [deg]'].map(direction_mapping)
ON_OFF_selectivity_data = ON_OFF_selectivity_data.drop(columns=["edge_intensity", "bg_intensity", 'rotation [deg]'])
ON_OFF_selectivity_data = ON_OFF_selectivity_data.sort_values(['genotype_sort', 'unique_roi', 'time']).reset_index(drop=True)

In [ ]:
all_processed_data = pd.concat([dir_tuning_data, temp_tuning_data, flashes_data, ON_OFF_selectivity_data])
all_processed_data = all_processed_data.sort_values(['genotype_sort', 'unique_roi', 'time'])

In [ ]:
quantify = dir_tuning_data.copy()

# Calculate min df/f values during stimulus time for every roi (all edge rotations taken together) and subtract from the full time traces
min_values_during_stimulus = pd.DataFrame(quantify.query("time > 0 and time < 5.18").groupby('unique_roi')['df/f'].min()).reset_index().rename(columns={'df/f':'min_value'})
min_values_during_stimulus_dict = dict(zip(min_values_during_stimulus.unique_roi, min_values_during_stimulus.min_value))
quantify['min_value_during_stim'] = quantify.unique_roi.map(min_values_during_stimulus_dict)
quantify['df/f_min_subt'] = quantify['df/f'] - quantify['min_value_during_stim']
quantify['rotation [rad]'] = np.deg2rad(quantify["rotation [deg]"])

# Calculate peak response for every roi for every stimulus 
# First create grouper unique_stimulus for unique stim/roi combinations
cols_to_factorize_on = ['unique_roi', 'roi_type', 'polarity', 'velocity [deg/s]', 'contrast', 'rotation [deg]']
quantify = factorize_dataset(quantify, cols_to_factorize_on, new_col_name="unique_stimulus")
quantify = quantify.sort_values(['genotype_sort', 'genotype', 'experiment', 'fly', 'filename_base', 'unique_roi','rotation [deg]', 'time']).reset_index(drop=True)

# For all unique stim/roi combinations, extract peak response time frame (within stimulus time 0 - 5.18s) by taking -1 to 2s around peak 
filtered = quantify.query("time > 0 and time < 5.18")
quantify_max_values = (filtered.loc[filtered.groupby("unique_stimulus")["df/f"].idxmax()].reset_index(drop=True))

peak_times = quantify_max_values[['unique_stimulus', 'time']].rename(columns={'time':'peak_time'})
peak_times = dict(zip(peak_times['unique_stimulus'], peak_times['peak_time']))
quantify['peak_time'] = quantify.apply(lambda row: peak_times[row['unique_stimulus']], axis=1)
quantify['relative_time'] = np.around(quantify['time'] - quantify['peak_time'], decimals=5)
quantify = quantify.query("relative_time > -1 and relative_time < 2").reset_index(drop=True)

In [ ]:
# Calculate vectors (x and y values) for all stimuli for each roi (for Ldir)
Ldir_quantify = quantify_max_values.copy()
Ldir_quantify['x'] = Ldir_quantify['df/f_min_subt'] * np.cos(Ldir_quantify['rotation [rad]'])
Ldir_quantify['y'] = Ldir_quantify['df/f_min_subt'] * np.sin(Ldir_quantify['rotation [rad]'])
Ldir_quantify['single_vec_lengths'] = Ldir_quantify.apply(lambda row: math.sqrt(row['x'] ** 2 + row['y'] ** 2), axis=1)

all_x = pd.DataFrame(Ldir_quantify.groupby(['unique_roi']).x.sum())
all_y = pd.DataFrame(Ldir_quantify.groupby(['unique_roi']).y.sum())
all_vectors = all_x.join(all_y).reset_index()

# Resulting vector (weighted average preferred direction (PD) of motion per ROI)
all_vectors["res_vec_rad"] = all_vectors.apply(lambda row: math.atan2(row['y'], row['x']), axis=1)
all_vectors["res_vec_deg"] = np.rad2deg(all_vectors["res_vec_rad"])

all_vectors["res_vec_mag"] = all_vectors.apply(lambda row: math.sqrt(row['x'] ** 2 + row['y'] ** 2), axis=1)
sum_subvecs = pd.DataFrame(Ldir_quantify.groupby("unique_roi").single_vec_lengths.sum()).rename(columns={'single_vec_lengths': 'sum_subvecs'})

all_vectors = pd.merge(left=all_vectors, right=sum_subvecs, on='unique_roi')
all_vectors["Ldir"] = all_vectors["res_vec_mag"] / all_vectors["sum_subvecs"]

Ldirs = all_vectors[["unique_roi", "Ldir"]].copy()
Ldir = quantify_max_values[["fly", "genotype", "roi_type", "unique_roi", 'polarity']].copy().drop_duplicates()
Ldir = pd.merge(Ldir, Ldirs, on='unique_roi')

Ldir["order"] = Ldir.genotype.map(lookup_genotype)
Ldir = Ldir.sort_values(['order', 'genotype', 'fly', 'roi_type', 'polarity'])

Ldir_flymeans = pd.DataFrame(Ldir.groupby(['order', 'genotype', 'fly', 'roi_type', 'polarity']).Ldir.mean()).reset_index()

In [ ]:
## Align all PD's to 0 degrees (front-to-back)
def calculate_closest_direction(list_of_points, target):
    # list_of_points: list of points on unit circle to compare the target with (in degrees)
    # target: in degrees
    
    # Distance between points on circle: https://stackoverflow.com/questions/36816284/find-closest-number-in-a-circular-array
    options = {}
    for point in list_of_points:
        distance = np.sqrt(2-2*(np.cos(np.deg2rad(target - point))))
        if point not in options:
            options[distance] = point
    return options[min(options)]

all_tested_rotations = quantify_max_values['rotation [deg]'].unique() # Use this only if all ROIs have been subjected to the same set of stimuli rotations

# For peak response data
all_vectors["measured_direction_closest_to_PD"] = all_vectors.apply(lambda row: calculate_closest_direction(all_tested_rotations, row.res_vec_deg), axis=1)
all_vectors["align_roll_value"] = all_vectors.apply(lambda row: int((row["measured_direction_closest_to_PD"] - 0) / 18), axis=1)

roi_align = dict(zip(all_vectors['unique_roi'], all_vectors['align_roll_value']))

quantify_max_values_aligned = quantify_max_values.copy()
quantify_max_values_aligned['align_roll_value'] = quantify_max_values_aligned.unique_roi.map(roi_align)
quantify_max_values_aligned['rotation_index_[deg]'] = (quantify_max_values_aligned['rotation [deg]'] / 18).astype(int)
quantify_max_values_aligned['rotation_aligned_[deg]'] = quantify_max_values_aligned.apply(lambda row: np.roll(all_tested_rotations, row['align_roll_value'])[row['rotation_index_[deg]']], axis=1)
quantify_max_values_aligned["rotation_aligned_[rad]"] = np.deg2rad(quantify_max_values_aligned["rotation_aligned_[deg]"])

# For time traces
quantify_aligned = quantify.copy()
quantify_aligned['align_roll_value'] = quantify_aligned.unique_roi.map(roi_align)
quantify_aligned['rotation_index_[deg]'] = (quantify_aligned['rotation [deg]'] / 18).astype(int)
quantify_aligned['rotation_aligned_[deg]'] = quantify_aligned.apply(lambda row: np.roll(all_tested_rotations, row['align_roll_value'])[row['rotation_index_[deg]']], axis=1)
quantify_aligned["rotation_aligned_[rad]"] = np.deg2rad(quantify_aligned["rotation_aligned_[deg]"])

In [ ]:
# Average ROIs of the same roi_type for each recording
# for quantify_aligned
grouper = ['experiment', 'genotype_sort', 'genotype', 'fly', 'filename_base', 'roi_type', 'name', 'start pause [sec]', 'rotation_aligned_[deg]', 'rotation_aligned_[rad]', 'velocity [deg/s]', 'polarity', 'contrast', 'relative_time']

plot_responses_flyaveraged = (pd.DataFrame(quantify_aligned.groupby(grouper)['df/f'].apply(lambda x: np.mean(x)))).rename(columns={'df/f':'df/f_averaged'}).reset_index()
plot_responses_flyaveraged = plot_responses_flyaveraged.merge(pd.DataFrame(plot_responses_flyaveraged.groupby(['experiment', 'genotype_sort', 'genotype', 'fly', 'filename_base', 'roi_type', 'name', 'start pause [sec]', 'velocity [deg/s]', 'polarity', 'contrast'])['df/f_averaged'].max()).rename(columns={'df/f_averaged':'df/f_max'}).reset_index())
plot_responses_flyaveraged['df/f_normalized'] = plot_responses_flyaveraged['df/f_averaged'] / plot_responses_flyaveraged['df/f_max']
plot_responses_flyaveraged = plot_responses_flyaveraged.sort_values(['experiment', 'genotype_sort', 'genotype', 'fly', 'filename_base', 'name', 'roi_type', 'rotation_aligned_[deg]', 'relative_time'])
plot_responses_flyaveraged

# for quantify_max_values_aligned
grouper = ['experiment', 'genotype_sort', 'genotype', 'fly', 'filename_base', 'roi_type', 'name', 'start pause [sec]', 'rotation_aligned_[deg]', 'rotation_aligned_[rad]', 'velocity [deg/s]', 'polarity', 'contrast']
plot_peaks_flyaveraged = (pd.DataFrame(quantify_max_values_aligned.groupby(grouper)['df/f_min_subt'].apply(lambda x: np.mean(x)))).rename(columns={'df/f_min_subt':'df/f_min_subt_averaged'}).reset_index()
plot_peaks_flyaveraged = plot_peaks_flyaveraged.merge(pd.DataFrame(plot_peaks_flyaveraged.groupby(['experiment', 'genotype_sort', 'genotype', 'fly', 'filename_base', 'roi_type', 'name', 'start pause [sec]', 'velocity [deg/s]', 'polarity', 'contrast'])['df/f_min_subt_averaged'].max()).rename(columns={'df/f_min_subt_averaged':'df/f_min_subt_max'}).reset_index())
plot_peaks_flyaveraged['df/f_min_subt_normalized'] = plot_peaks_flyaveraged['df/f_min_subt_averaged'] / plot_peaks_flyaveraged['df/f_min_subt_max']
plot_peaks_flyaveraged = plot_peaks_flyaveraged.sort_values(['experiment', 'genotype_sort', 'genotype', 'fly', 'filename_base', 'name', 'roi_type', 'rotation_aligned_[deg]'])

In [ ]:
### Create NEW list of all stimulus parameters for each stimulus type ###

non_stimulus_specs_2 = non_stimulus_specs + ['unique_roi', 'start pause [sec]', 'experiment']
stimuli_2 = []
column_names_2 = []

for experiment, dataset in all_processed_data.groupby('experiment'):
    stimuli_2.append(experiment)
    column_names_2.append(dataset.dropna(how='any', axis=1).columns.to_list())
    
stimuli_to_loop_2 = pd.DataFrame({'experiment': stimuli_2, 'column_names': column_names_2})

stimuli_to_loop_2 = stimuli_to_loop_2.copy()
stimuli_to_loop_2['stim_params'] = stimuli_to_loop_2['column_names'].apply(lambda row: get_stim_params(row, non_stimulus_specs_2))
stimuli_to_loop_2 = stimuli_to_loop_2.set_index('experiment')
stimuli_to_loop_2.at['dir_tuning', 'stim_params'].remove('rotation [deg]') 
stimuli_to_loop_2.at['dir_tuning', 'stim_params'] = stimuli_to_loop_2.at['dir_tuning', 'stim_params'] + ['rotation_aligned_[deg]']

In [ ]:
data_n = pd.DataFrame(pd.DataFrame(plot_responses_flyaveraged.groupby(['experiment', 'genotype_sort', 'genotype', 'fly', 'filename_base', 'roi_type', 'polarity'])['rotation_aligned_[deg]'].agg(list)))
flies_n = pd.DataFrame(data_n.reset_index().groupby(['experiment', 'genotype_sort', 'genotype', 'polarity', 'roi_type']).fly.unique())
flies_n['n_flies'] = flies_n.fly.apply(len)
flies_n

# Flash response analysis

In [ ]:
flashes_data_quantify_ON = pd.DataFrame(flashes_data.query("roi_type in ['med_whole', 'lob_whole'] and time > 0 and time < 0.3").groupby(['genotype_sort', 'genotype', 'fly', 'filename_base', 'roi_type', 'unique_roi'])['df/f'].max()).rename(columns={'df/f': 'flash_response'}).reset_index()
flashes_data_quantify_ON['flash_polarity'] = 'ON'
flashes_data_quantify_OFF = pd.DataFrame(flashes_data.query("roi_type in ['med_whole', 'lob_whole'] and time > 2 and time < 2.3").groupby(['genotype_sort', 'genotype', 'fly', 'filename_base', 'roi_type', 'unique_roi'])['df/f'].max()).rename(columns={'df/f': 'flash_response'}).reset_index()
flashes_data_quantify_OFF['flash_polarity'] = 'OFF'
flashes_data_quantify = pd.concat([flashes_data_quantify_ON, flashes_data_quantify_OFF]).reset_index(drop=True)
flashes_data_quantify['order_genotype'] = flashes_data_quantify.apply(lambda row: str(row.genotype_sort) + '_' + row.genotype, axis=1)

flashes_data_quantify_avg = pd.DataFrame(flashes_data_quantify.groupby(['genotype', 'roi_type', 'flash_polarity']).flash_response.mean()).reset_index()
flashes_data_quantify.groupby(['genotype', 'roi_type', 'flash_polarity',]).fly.count()

In [ ]:
df = flashes_data.query("roi_type in ['med_whole', 'lob_whole'] and genotype in ['T4T5ad', 'HisCl1_RNAi', 'Ort_RNAi', 'Rdl_RNAi', 'Rdl_Ort_RNAi']")
rows = ['genotype_sort','genotype']

stimuli_loop = []
stim_n_loop = []

for name, stim in df.groupby('experiment'):
    stim_n = stim.groupby(stimuli_to_loop_2.loc[name,'stim_params']).ngroups
    stimuli_loop.append(name)
    stim_n_loop.append(stim_n)

stimuli_to_loop_df = pd.DataFrame({'experiment': stimuli_loop, 'columns_n': stim_n_loop})
cols_n = 6

rows_iter = df.groupby(rows, sort=False)
rows_n = 2

fig, ax = plt.subplot_mosaic(
      '''
    AACCEEGGII
    BBDDFFHHJJ
    KKKKKLLLLL
    ''',
    sharex=False,
    sharey=False,
    figsize=(2.4, 2.05),
    height_ratios=[0.5,0.5,1],
)

# row_count = 0
# col_count = 0
ax_id_iter = 'ABCDEFGHIJ'
ax_iter = 0

for name_acq, row in rows_iter:
    for name_stim, cols_stim_type in row.groupby('experiment'):
        cols_stim_type = cols_stim_type.sort_values('roi_type', ascending=False)
        for name_params, cols_stim_params in cols_stim_type.groupby(['genotype'] + ['roi_type'], sort=False):
            genotype = cols_stim_params.genotype.unique()[0]
            area = cols_stim_params.roi_type.unique()[0]
            n_flies = len(cols_stim_params.fly.unique()[:])
            ax_id = ax[ax_id_iter[ax_iter]]
            
            sns.lineplot(data=cols_stim_params, x='time', y='df/f', ax=ax_id, color=palette_genotypes[cols_stim_params.genotype.unique()[0]], estimator='mean', linewidth = 1, errorbar='sd', err_kws={'linewidth': 0})
            ax_id.set_ylabel('ΔF/F')
            ax_id.set_xlabel('Time (s)')
            if area == 'med_whole':
                ax_id.set_xlim([-0.5, 1])
            else:
                ax_id.set_xlim([1.5, 3])
                ax_id.set_xticks([2,3], labels=[0,1])
            
            ax_id.set_ylim([-0.15, 1.5])
            ax_id.set_yticks([0, 1])
            ax_id.fill_between([2,3.9], 2.1, -0.15, color='k', alpha=0.08, lw=0)
            ax_id.fill_between([-0.5,0], 2.1, -0.15, color='k', alpha=0.08, lw=0)

            ax_id.yaxis.set_visible(False)
            ax_id.xaxis.set_visible(False)
            ax_id.spines[['top', 'right', 'left', 'bottom']].set_visible(False)

            ax_iter = ax_iter + 1

ax['B'].yaxis.set_visible(True) 
ax['B'].xaxis.set_visible(True) 
ax['B'].spines[['left', 'bottom']].set_visible(True)
            
sns.stripplot(data=flashes_data_quantify.query("roi_type == 'med_whole' and flash_polarity == 'ON'"), x="genotype", y="flash_response", hue="genotype", palette=palette_genotypes, alpha=0.3, s=3, jitter=0.12, dodge=False, edgecolor='white', linewidth=0.2, ax=ax['K'], legend=None, order=['T4T5ad', 'HisCl1_RNAi', 'Rdl_RNAi', 'Ort_RNAi','Rdl_Ort_RNAi']);
sns.scatterplot(data=flashes_data_quantify_avg.query("roi_type == 'med_whole' and flash_polarity == 'ON'"), x="genotype", y="flash_response", hue="genotype", palette=palette_genotypes, alpha=1, ax=ax['K'], legend=None, zorder=3, marker='_', s=80, lw=1.5);

sns.stripplot(data=flashes_data_quantify.query("roi_type == 'lob_whole' and flash_polarity == 'OFF'"), x="genotype", y="flash_response", hue="genotype", palette=palette_genotypes,  alpha=0.3, s=3, jitter=0.12, dodge=False, edgecolor='white', linewidth=0.2, ax=ax['L'], legend=None, order=['T4T5ad', 'HisCl1_RNAi', 'Rdl_RNAi', 'Ort_RNAi', 'Rdl_Ort_RNAi']);
sns.scatterplot(data=flashes_data_quantify_avg.query("roi_type == 'lob_whole' and flash_polarity == 'OFF'"), x="genotype", y="flash_response", hue="genotype", palette=palette_genotypes, alpha=1, ax=ax['L'], legend=None, zorder=3, marker='_', s=80, lw=1.5);

for plot in 'KL':
    ax[plot].set_ylabel('Response ΔF/F')
    ax[plot].set_xlabel('')
    ax[plot].set_ylim([-0.05,2.2])
    ax[plot].margins(x=0.1)
    ax[plot].spines[['top', 'right']].set_visible(False)

ax['L'].yaxis.set_visible(False) 

# plt.subplots_adjust(hspace=0.5)
plt.subplots_adjust(wspace=0.5)

figtitle = "flashes_traces_mean_sd.pdf"
plt.savefig(savepath_fig / figtitle)

In [ ]:
# For statistics in Graphpad Prism
flashes_data_graphpad_med = flashes_data_quantify.query("roi_type == 'med_whole' and flash_polarity == 'ON'").pivot(columns='order_genotype', values='flash_response').reset_index(drop=True)
flashes_data_graphpad_med = flashes_data_graphpad_med.apply(lambda x: sorted(x, key=pd.isnull), 0).dropna(how='all')
flashes_data_graphpad_med.to_csv(savepath_data / 'T4_max_flash_response.csv')

flashes_data_graphpad_lob = flashes_data_quantify.query("roi_type == 'lob_whole' and flash_polarity == 'OFF'").pivot(columns='order_genotype', values='flash_response').reset_index(drop=True)
flashes_data_graphpad_lob = flashes_data_graphpad_lob.apply(lambda x: sorted(x, key=pd.isnull), 0).dropna(how='all')
flashes_data_graphpad_lob.to_csv(savepath_data / 'T5_max_flash_response.csv')
# flashes_data_graphpad_med.mean()
# flashes_data_graphpad_med.sem()
# flashes_data_graphpad_lob.mean()
# flashes_data_graphpad_lob.sem()

# Ldir analysis bright and dark edges

In [ ]:
# Responses to bright moving edges in LP1 (predominantly T4)
cm_inch = 1/2.54 

df = plot_responses_flyaveraged.query("experiment == 'dir_tuning' and polarity == 1 and roi_type == 'lp_layer1'")
rows = ['genotype_sort','genotype']

plot_order = dict(zip(np.roll(all_tested_rotations, 10), np.arange(0,20)))

stimuli_loop = []
stim_n_loop = []

for name, stim in df.groupby('experiment'):
    stim_n = stim.groupby(stimuli_to_loop_2.loc[name,'stim_params']).ngroups
    stimuli_loop.append(name)
    stim_n_loop.append(stim_n)

stimuli_to_loop_df = pd.DataFrame({'experiment': stimuli_loop, 'columns_n': stim_n_loop})
cols_n = stimuli_to_loop_df.columns_n.values.sum()

rows_iter = df.groupby(rows, sort=False)
rows_n = rows_iter.ngroups

fig, axs = plt.subplots(rows_n, cols_n, figsize=(10 * cm_inch, 5 * cm_inch), sharex=True, sharey=True, squeeze=False, dpi=300)

row_count = 0

for name_acq, row in rows_iter:
    for name_stim, cols_stim_type in row.groupby('experiment'):
        for name_params, cols_stim_params in cols_stim_type.groupby(stimuli_to_loop_2.loc[name_stim,'stim_params']):
            n_flies = len(cols_stim_params.fly.unique())
            col_nr = plot_order[cols_stim_params['rotation_aligned_[deg]'].unique()[0]]
            sns.lineplot(data=cols_stim_params, x='relative_time', y='df/f_normalized', ax=axs[row_count,col_nr], color=palette_genotypes[cols_stim_params.genotype.unique()[0]], alpha = 1, estimator='mean', errorbar='sd', err_kws={'linewidth': 0}, lw=0.75)
            axs[row_count,col_nr].set_ylabel('Norm. ΔF/F')
            axs[row_count,col_nr].set_xlabel('Time (s)')
            axs[row_count, col_nr].set_xlim([-0.8, 1.5])
            axs[row_count, col_nr].set_ylim([-0.17,1.07])

            if col_nr == 0:
                axs[row_count, 0].text(-0.7, 0.7, genotype_formatting[name_acq[1]] + '\nn = ' + str(n_flies), color=palette_genotypes[cols_stim_params.genotype.unique()[0]], fontsize=6, zorder=100)
    
            axs[row_count, col_nr].yaxis.set_visible(False) 
            axs[row_count, col_nr].xaxis.set_visible(False) 
            axs[row_count, col_nr].spines[['top', 'right', 'left', 'bottom']].set_visible(False)
            if row_count == 0:
                axs[row_count, col_nr].set_title(str(int(name_params[-1])), fontsize=6)
    row_count = row_count + 1

axs[4,0].yaxis.set_visible(True) 
axs[4,0].xaxis.set_visible(True) 
axs[4,0].spines[['bottom', 'left']].set_visible(True)

figtitle = "T4_bright_edges_normalized_mean_sd.pdf"
plt.savefig(savepath_fig / figtitle)

In [ ]:
# Responses to dark moving edges in LP1 (predominantly T5)
cm_inch = 1/2.54 

df = plot_responses_flyaveraged.query("experiment == 'dir_tuning' and polarity == -1 and roi_type == 'lp_layer1'")
rows = ['genotype_sort','genotype']

plot_order = dict(zip(np.roll(all_tested_rotations, 10), np.arange(0,20)))

stimuli_loop = []
stim_n_loop = []

for name, stim in df.groupby('experiment'):
    stim_n = stim.groupby(stimuli_to_loop_2.loc[name,'stim_params']).ngroups
    stimuli_loop.append(name)
    stim_n_loop.append(stim_n)

stimuli_to_loop_df = pd.DataFrame({'experiment': stimuli_loop, 'columns_n': stim_n_loop})
cols_n = stimuli_to_loop_df.columns_n.values.sum()

rows_iter = df.groupby(rows, sort=False)
rows_n = rows_iter.ngroups

fig, axs = plt.subplots(rows_n, cols_n, figsize=(10 * cm_inch, 5 * cm_inch), sharex=True, sharey=True, squeeze=False, dpi=300)

row_count = 0

for name_acq, row in rows_iter:
    for name_stim, cols_stim_type in row.groupby('experiment'):
        for name_params, cols_stim_params in cols_stim_type.groupby(stimuli_to_loop_2.loc[name_stim,'stim_params']):
            n_flies = len(cols_stim_params.fly.unique())
            col_nr = plot_order[cols_stim_params['rotation_aligned_[deg]'].unique()[0]]
            sns.lineplot(data=cols_stim_params, x='relative_time', y='df/f_normalized', ax=axs[row_count,col_nr], color=palette_genotypes[cols_stim_params.genotype.unique()[0]], alpha = 1, estimator='mean', errorbar='sd', err_kws={'linewidth': 0}, lw=0.75)
            axs[row_count,col_nr].set_ylabel('Norm. ΔF/F')
            axs[row_count,col_nr].set_xlabel('Time (s)')
            axs[row_count, col_nr].set_xlim([-0.8, 1.5])
            axs[row_count, col_nr].set_ylim([-0.17,1.07])

            if col_nr == 0:
                axs[row_count, 0].text(-0.7, 0.7, genotype_formatting[name_acq[1]] + '\nn = ' + str(n_flies), color=palette_genotypes[cols_stim_params.genotype.unique()[0]], fontsize=6, zorder=100)
    
            axs[row_count, col_nr].yaxis.set_visible(False) 
            axs[row_count, col_nr].xaxis.set_visible(False) 
            axs[row_count, col_nr].spines[['top', 'right', 'left', 'bottom']].set_visible(False)
            if row_count == 0:
                axs[row_count, col_nr].set_title(str(int(name_params[-1])), fontsize=6)
    row_count = row_count + 1

axs[4,0].yaxis.set_visible(True) 
axs[4,0].xaxis.set_visible(True) 
axs[4,0].spines[['bottom', 'left']].set_visible(True)

figtitle = "T5_dark_edges_normalized_mean_sd.pdf"
plt.savefig(savepath_fig / figtitle)

In [ ]:
plot_subgroups = plot_peaks_flyaveraged.query("roi_type == 'lp_layer1' and polarity == 1")
Ldir_plotting = Ldir_flymeans.query("roi_type == 'lp_layer1' and polarity == 1")
Ldir_plotting_means = pd.DataFrame(Ldir_plotting.groupby(['genotype'], sort=False).Ldir.mean()).reset_index()

plot_subgroups_mean = pd.DataFrame(plot_subgroups.groupby(['genotype', 'roi_type', 'polarity', 'rotation_aligned_[deg]', 'rotation_aligned_[rad]'])['df/f_min_subt_normalized'].mean()).rename(columns={'df/f_min_subt_normalized':'mean'})
plot_subgroups_sem = pd.DataFrame(plot_subgroups.groupby(['genotype', 'roi_type', 'polarity', 'rotation_aligned_[deg]', 'rotation_aligned_[rad]'])['df/f_min_subt_normalized'].sem()).rename(columns={'df/f_min_subt_normalized':'sem'})
plot_subgroups_n = pd.DataFrame(plot_subgroups.groupby(['genotype', 'roi_type', 'polarity']).fly.unique())
plot_subgroups_n = plot_subgroups_n.apply(lambda x: len(x.fly), axis=1).rename('n_flies').reset_index()

plot_subgroups = pd.concat([plot_subgroups_mean, plot_subgroups_sem], axis=1).reset_index()
genotype_max = pd.DataFrame(plot_subgroups.reset_index().groupby(['genotype', 'roi_type', 'polarity'])['mean'].max()).reset_index().rename(columns={'mean':'genotype_max'})
plot_subgroups = plot_subgroups.merge(genotype_max)
plot_subgroups = plot_subgroups.merge(plot_subgroups_n)
plot_subgroups['mean_genotype_normalized'] = plot_subgroups['mean'] / plot_subgroups['genotype_max']
plot_subgroups['sem_genotype_normalized'] = plot_subgroups['sem'] / plot_subgroups['genotype_max']

plotting_parameters_polar = [
    {'genotype': 'T4T5ad', 'axis': 'A','title': 'Bright edges LP'},
    {'genotype': 'HisCl1_RNAi', 'axis': 'A', 'title': 'Bright edges LP'},
    
    {'genotype': 'T4T5ad', 'axis': 'B','title': 'Bright edges LP'},
    {'genotype': 'Ort_RNAi', 'axis': 'B', 'title': 'Bright edges LP'},

    {'genotype': 'T4T5ad', 'axis': 'C', 'title': 'Bright edges LP'},
    {'genotype': 'Rdl_RNAi', 'axis': 'C', 'title': 'Bright edges LP'},

    {'genotype': 'T4T5ad', 'axis': 'D', 'title': 'Bright edges LP'},
    {'genotype': 'Rdl_Ort_RNAi', 'axis': 'D', 'title': 'Bright edges LP'},
]

fig, axd = plt.subplot_mosaic(
    'ABCDE',
    per_subplot_kw={
        "ABCD": {"projection": "polar"},
    },
    sharex=False,
    sharey=False,
    figsize=(21 * cm_inch, 4.2 * cm_inch),
    dpi=300,
    gridspec_kw={
        "wspace": 0.2,
        "hspace": 0.2,
    },
)

for subgroup in plotting_parameters_polar:
    genotype = subgroup['genotype']
    axis = subgroup['axis']
    title = genotype + '\n' + subgroup['title']
    n_flies = plot_subgroups.query("genotype == @genotype").n_flies.unique()[:][0]

    subgroup_data = plot_subgroups.query("genotype == @genotype")
    subgroup_data = pd.concat([subgroup_data, subgroup_data.query("`rotation_aligned_[deg]` == 0.0").assign(**{'rotation_aligned_[deg]': 360.0, 'rotation_aligned_[rad]': 2*np.pi})])
    axd[axis].plot(subgroup_data['rotation_aligned_[rad]'], subgroup_data['mean_genotype_normalized'], alpha = 1, color=palette_genotypes[genotype], lw=1)
    axd[axis].fill_between(subgroup_data['rotation_aligned_[rad]'], subgroup_data['mean_genotype_normalized']+subgroup_data['sem_genotype_normalized'], subgroup_data['mean_genotype_normalized']-subgroup_data['sem_genotype_normalized'], facecolor=palette_genotypes[genotype], alpha=0.3)
   
    if axis == 'A':
        if genotype == 'T4T5ad':
            axd[axis].text(0.75*np.pi, 1.5, genotype_formatting[genotype] + '\nn = ' + str(n_flies), color=palette_genotypes[genotype], fontsize=6, zorder=100)
        else: 
            axd[axis].text(0.85*np.pi, 1.5, genotype_formatting[genotype] + '\nn = ' + str(n_flies), color=palette_genotypes[genotype], fontsize=6, zorder=100)
    else:
        if genotype != 'T4T5ad':
            axd[axis].text(0.85*np.pi, 1.5, genotype_formatting[genotype] + '\nn = ' + str(n_flies), color=palette_genotypes[genotype], fontsize=6, zorder=100)
        
        
    axd[axis].set_ylim([0,1.1])
    axd[axis].set_yticks([0, 0.25, 0.5, 0.75, 1])
    axd[axis].set_yticklabels(['', '', '', '', ''], fontsize=0)
    axd[axis].grid(linewidth=0.5, linestyle='dashed')
    
    axd[axis].set_xticks([0, 0.25*np.pi, 0.5*np.pi, 0.75*np.pi, np.pi, 1.25*np.pi, 1.5*np.pi, 1.75*np.pi])
    axd[axis].set_xticklabels(['0°', '', '90°', '', '180°', '', '270°', ''])
    axd[axis].tick_params(axis='x', which='major', pad=-2.5)

### Ldir plots!
dotsize = 3

a = sns.stripplot(x="genotype", y="Ldir", hue="genotype", palette=palette_genotypes, 
                  data=Ldir_plotting.query("genotype in ['T4T5ad', 'HisCl1_RNAi', 'Ort_RNAi', 'Rdl_RNAi', 'Rdl_Ort_RNAi']"), dodge=False, alpha=0.3, size=dotsize, edgecolor='white', linewidth=0.2, ax=axd['E'])
sns.scatterplot(data=Ldir_plotting_means.query("genotype in ['T4T5ad', 'HisCl1_RNAi', 'Ort_RNAi', 'Rdl_RNAi', 'Rdl_Ort_RNAi']"), x='genotype', y='Ldir', hue='genotype', palette=palette_genotypes, alpha=1, ax=axd['E'], legend=None, zorder=3, marker='_', s=70, lw=1.5)
a.set_ylim([0, 0.51])
axd['E'].spines[['top', 'right']].set_visible(False)

figtitle = "T4_polarplots_normalized_mean_sem_Ldir.pdf"
plt.savefig(savepath_fig / figtitle)

In [ ]:
plot_subgroups = plot_peaks_flyaveraged.query("roi_type == 'lp_layer1' and polarity == -1")
Ldir_plotting = Ldir_flymeans.query("roi_type == 'lp_layer1' and polarity == -1")
Ldir_plotting_means = pd.DataFrame(Ldir_plotting.groupby(['genotype'], sort=False).Ldir.mean()).reset_index()

plot_subgroups_mean = pd.DataFrame(plot_subgroups.groupby(['genotype', 'roi_type', 'polarity', 'rotation_aligned_[deg]', 'rotation_aligned_[rad]'])['df/f_min_subt_normalized'].mean()).rename(columns={'df/f_min_subt_normalized':'mean'})
plot_subgroups_sem = pd.DataFrame(plot_subgroups.groupby(['genotype', 'roi_type', 'polarity', 'rotation_aligned_[deg]', 'rotation_aligned_[rad]'])['df/f_min_subt_normalized'].sem()).rename(columns={'df/f_min_subt_normalized':'sem'})
plot_subgroups_n = pd.DataFrame(plot_subgroups.groupby(['genotype', 'roi_type', 'polarity']).fly.unique())
plot_subgroups_n = plot_subgroups_n.apply(lambda x: len(x.fly), axis=1).rename('n_flies').reset_index()

plot_subgroups = pd.concat([plot_subgroups_mean, plot_subgroups_sem], axis=1).reset_index()
genotype_max = pd.DataFrame(plot_subgroups.reset_index().groupby(['genotype', 'roi_type', 'polarity'])['mean'].max()).reset_index().rename(columns={'mean':'genotype_max'})
plot_subgroups = plot_subgroups.merge(genotype_max)
plot_subgroups = plot_subgroups.merge(plot_subgroups_n)
plot_subgroups['mean_genotype_normalized'] = plot_subgroups['mean'] / plot_subgroups['genotype_max']
plot_subgroups['sem_genotype_normalized'] = plot_subgroups['sem'] / plot_subgroups['genotype_max']

plotting_parameters_polar = [
    {'genotype': 'T4T5ad', 'axis': 'A','title': 'Bright edges LP'},
    {'genotype': 'HisCl1_RNAi', 'axis': 'A', 'title': 'Bright edges LP'},
    
    {'genotype': 'T4T5ad', 'axis': 'B','title': 'Bright edges LP'},
    {'genotype': 'Ort_RNAi', 'axis': 'B', 'title': 'Bright edges LP'},

    {'genotype': 'T4T5ad', 'axis': 'C', 'title': 'Bright edges LP'},
    {'genotype': 'Rdl_RNAi', 'axis': 'C', 'title': 'Bright edges LP'},

    {'genotype': 'T4T5ad', 'axis': 'D', 'title': 'Bright edges LP'},
    {'genotype': 'Rdl_Ort_RNAi', 'axis': 'D', 'title': 'Bright edges LP'},
]

fig, axd = plt.subplot_mosaic(
    'ABCDE',
    per_subplot_kw={
        "ABCD": {"projection": "polar"},
    },
    sharex=False,
    sharey=False,
    figsize=(21 * cm_inch, 4.2 * cm_inch),
    dpi=300,
    gridspec_kw={
        "wspace": 0.2,
        "hspace": 0.2,
    },
)

for subgroup in plotting_parameters_polar:
    genotype = subgroup['genotype']
    axis = subgroup['axis']
    title = genotype + '\n' + subgroup['title']
    n_flies = plot_subgroups.query("genotype == @genotype").n_flies.unique()[:][0]

    subgroup_data = plot_subgroups.query("genotype == @genotype")
    subgroup_data = pd.concat([subgroup_data, subgroup_data.query("`rotation_aligned_[deg]` == 0.0").assign(**{'rotation_aligned_[deg]': 360.0, 'rotation_aligned_[rad]': 2*np.pi})])
    axd[axis].plot(subgroup_data['rotation_aligned_[rad]'], subgroup_data['mean_genotype_normalized'], alpha = 1, color=palette_genotypes[genotype], lw=1)
    axd[axis].fill_between(subgroup_data['rotation_aligned_[rad]'], subgroup_data['mean_genotype_normalized']+subgroup_data['sem_genotype_normalized'], subgroup_data['mean_genotype_normalized']-subgroup_data['sem_genotype_normalized'], facecolor=palette_genotypes[genotype], alpha=0.3)
   
    if axis == 'A':
        if genotype == 'T4T5ad':
            axd[axis].text(0.75*np.pi, 1.5, genotype_formatting[genotype] + '\nn = ' + str(n_flies), color=palette_genotypes[genotype], fontsize=6, zorder=100)
        else: 
            axd[axis].text(0.85*np.pi, 1.5, genotype_formatting[genotype] + '\nn = ' + str(n_flies), color=palette_genotypes[genotype], fontsize=6, zorder=100)
    else:
        if genotype != 'T4T5ad':
            axd[axis].text(0.85*np.pi, 1.5, genotype_formatting[genotype] + '\nn = ' + str(n_flies), color=palette_genotypes[genotype], fontsize=6, zorder=100)
        
        
    axd[axis].set_ylim([0,1.1])
    axd[axis].set_yticks([0, 0.25, 0.5, 0.75, 1])
    axd[axis].set_yticklabels(['', '', '', '', ''], fontsize=0)
    axd[axis].grid(linewidth=0.2, linestyle='dashed')
    
    axd[axis].set_xticks([0, 0.25*np.pi, 0.5*np.pi, 0.75*np.pi, np.pi, 1.25*np.pi, 1.5*np.pi, 1.75*np.pi])
    axd[axis].set_xticklabels(['0°', '', '90°', '', '180°', '', '270°', ''])
    axd[axis].tick_params(axis='x', which='major', pad=-2.5)

### Ldir plots!


dotsize = 3

a = sns.stripplot(x="genotype", y="Ldir", hue="genotype", palette=palette_genotypes, 
                  data=Ldir_plotting.query("genotype in ['T4T5ad', 'HisCl1_RNAi', 'Ort_RNAi', 'Rdl_RNAi', 'Rdl_Ort_RNAi']"), dodge=False, alpha=0.3, size=dotsize, edgecolor='white', linewidth=0.2, ax=axd['E'])
sns.scatterplot(data=Ldir_plotting_means.query("genotype in ['T4T5ad', 'HisCl1_RNAi', 'Ort_RNAi', 'Rdl_RNAi', 'Rdl_Ort_RNAi']"), x='genotype', y='Ldir', hue='genotype', palette=palette_genotypes, alpha=1, ax=axd['E'], legend=None, zorder=3, marker='_', s=70, lw=1.5)
a.set_ylim([0, 0.51])
axd['E'].spines[['top', 'right']].set_visible(False)

figtitle = "T5_polarplots_normalized_mean_sem_Ldir.pdf"
plt.savefig(savepath_fig / figtitle)

In [ ]:
# For statistics in Graphpad Prism
Ldir_T4 = Ldir_flymeans.query("roi_type == 'lp_layer1' and polarity == 1")
Ldir_T4['order_genotype'] = Ldir_T4.apply(lambda row: str(row.order) + '_' + row.genotype, axis=1)
Ldir_T4_wide = Ldir_T4.pivot(index='fly', columns='order_genotype', values='Ldir').reset_index().drop(columns='fly')
Ldir_T4_wide = Ldir_T4_wide.apply(lambda x: sorted(x, key=pd.isnull), 0).dropna(how='all')
Ldir_T4_wide.to_csv(savepath_data / 'Ldir_T4.csv', sep=',', float_format='%.5f')

Ldir_T5 = Ldir_flymeans.query("roi_type == 'lp_layer1' and polarity == -1")
Ldir_T5['order_genotype'] = Ldir_T5.apply(lambda row: str(row.order) + '_' + row.genotype, axis=1)
Ldir_T5_wide = Ldir_T5.pivot(index='fly', columns='order_genotype', values='Ldir').reset_index().drop(columns='fly')
Ldir_T5_wide = Ldir_T5_wide.apply(lambda x: sorted(x, key=pd.isnull), 0).dropna(how='all')
Ldir_T5_wide.to_csv(savepath_data / 'Ldir_T5.csv', sep=',', float_format='%.5f')

# ON - OFF selectivity to moving edges in PD

In [ ]:
quantify_ON_OFF = ON_OFF_selectivity_data.copy()

# Calculate peak response for every roi for every stimulus 
# First create grouper unique_stimulus for unique stim/roi combinations
cols_to_factorize_on_ON_OFF = ['unique_roi', 'roi_type', 'polarity', 'velocity [deg/s]', 'contrast', 'direction']
quantify_ON_OFF = factorize_dataset(quantify_ON_OFF, cols_to_factorize_on_ON_OFF, new_col_name="unique_stimulus")
quantify_ON_OFF = quantify_ON_OFF.sort_values(['genotype_sort', 'genotype', 'experiment', 'fly', 'filename_base', 'unique_roi', 'direction', 'time']).reset_index(drop=True)

# For all unique stim/roi combinations, extract peak response time frame (within stimulus time 0 - 5.18s) by taking -1 to 2s around peak 
filtered_ON_OFF = quantify_ON_OFF.query("time > 0 and time < 5.18")
quantify_max_values_ON_OFF = (filtered_ON_OFF.loc[filtered_ON_OFF.groupby("unique_stimulus")["df/f"].idxmax()].reset_index(drop=True))

peak_times_ON_OFF = quantify_max_values_ON_OFF[['unique_stimulus', 'time']].rename(columns={'time':'peak_time'})
peak_times_ON_OFF = dict(zip(peak_times_ON_OFF['unique_stimulus'], peak_times_ON_OFF['peak_time']))
quantify_ON_OFF['peak_time'] = quantify_ON_OFF.apply(lambda row: peak_times_ON_OFF[row['unique_stimulus']], axis=1)
quantify_ON_OFF['relative_time'] = np.around(quantify_ON_OFF['time'] - quantify_ON_OFF['peak_time'], decimals=5)
quantify_ON_OFF = quantify_ON_OFF.query("relative_time > -1 and relative_time < 2").reset_index(drop=True)

grouper_ON_OFF = ['experiment', 'genotype_sort', 'genotype', 'fly', 'filename_base', 'roi_type', 'name', 'start pause [sec]', 'direction', 'velocity [deg/s]', 'polarity', 'contrast', 'relative_time']

# Average ROIs of the same roi_type for each recording
plot_responses_flyaveraged_ON_OFF = (pd.DataFrame(quantify_ON_OFF.groupby(grouper_ON_OFF)['df/f'].apply(lambda x: np.mean(x)))).rename(columns={'df/f':'df/f_averaged'}).reset_index()
plot_responses_flyaveraged_ON_OFF = plot_responses_flyaveraged_ON_OFF.merge(pd.DataFrame(plot_responses_flyaveraged_ON_OFF.groupby(['experiment', 'genotype_sort', 'genotype', 'fly', 'filename_base', 'roi_type', 'name', 'start pause [sec]', 'velocity [deg/s]', 'contrast'])['df/f_averaged'].max()).rename(columns={'df/f_averaged':'df/f_max'}).reset_index())
plot_responses_flyaveraged_ON_OFF['df/f_normalized'] = plot_responses_flyaveraged_ON_OFF['df/f_averaged'] / plot_responses_flyaveraged_ON_OFF['df/f_max']
plot_responses_flyaveraged_ON_OFF = plot_responses_flyaveraged_ON_OFF.sort_values(['experiment', 'genotype_sort', 'genotype', 'fly', 'filename_base', 'name', 'roi_type', 'direction', 'relative_time']).reset_index(drop=True)

In [ ]:
plot_ON_OFF = plot_responses_flyaveraged_ON_OFF.query("roi_type in ['med', 'lob']")
plot_ON_quantify = plot_ON_OFF.query("relative_time == 0 and polarity == 1").reset_index(drop=True)
plot_ON_quantify['polarity'] = 'ON'
plot_OFF_quantify = plot_ON_OFF.query("relative_time == 0 and polarity == -1").reset_index(drop=True)
plot_OFF_quantify['polarity'] = 'OFF'
plot_ON_OFF_quantify = pd.concat([plot_ON_quantify, plot_OFF_quantify])

ON_OFF_graphpad = plot_ON_OFF_quantify[['genotype_sort', 'genotype', 'fly', 'roi_type', 'polarity', 'df/f_averaged']].copy()
ON_OFF_graphpad['order_genotype'] = ON_OFF_graphpad.apply(lambda row: str(row.genotype_sort) + '_' + row.genotype, axis=1)
ON_OFF_graphpad = ON_OFF_graphpad.drop(columns=['genotype_sort'])
ON_OFF_graphpad_med_ON = ON_OFF_graphpad.query("roi_type == 'med' and polarity == 'ON'").rename(columns={'df/f_averaged': 'ON_response'}).drop(columns='polarity')
ON_OFF_graphpad_med_OFF = ON_OFF_graphpad.query("roi_type == 'med' and polarity == 'OFF'").rename(columns={'df/f_averaged': 'OFF_response'}).drop(columns='polarity')
ON_OFF_graphpad_lob_ON = ON_OFF_graphpad.query("roi_type == 'lob' and polarity == 'ON'").rename(columns={'df/f_averaged': 'ON_response'}).drop(columns='polarity')
ON_OFF_graphpad_lob_OFF = ON_OFF_graphpad.query("roi_type == 'lob' and polarity == 'OFF'").rename(columns={'df/f_averaged': 'OFF_response'}).drop(columns='polarity')

ON_OFF_graphpad_med = ON_OFF_graphpad_med_ON.merge(ON_OFF_graphpad_med_OFF)
ON_OFF_graphpad_lob = ON_OFF_graphpad_lob_ON.merge(ON_OFF_graphpad_lob_OFF)

ON_OFF_graphpad_med['ON_OFF_selectivity'] = (ON_OFF_graphpad_med['ON_response'] - ON_OFF_graphpad_med['OFF_response']) / (ON_OFF_graphpad_med['ON_response'] + ON_OFF_graphpad_med['OFF_response'])
ON_OFF_graphpad_lob['ON_OFF_selectivity'] = (ON_OFF_graphpad_lob['ON_response'] - ON_OFF_graphpad_lob['OFF_response']) / (ON_OFF_graphpad_lob['ON_response'] + ON_OFF_graphpad_lob['OFF_response'])

ON_OFF_graphpad = pd.concat([ON_OFF_graphpad_med, ON_OFF_graphpad_lob])
ON_OFF_graphpad_means = pd.DataFrame(ON_OFF_graphpad.groupby(['order_genotype', 'genotype', 'roi_type']).ON_OFF_selectivity.mean()).reset_index()

ON_OFF_graphpad_med_wide = ON_OFF_graphpad_med.pivot(index='fly', columns='order_genotype', values='ON_OFF_selectivity').reset_index().drop(columns='fly')
ON_OFF_graphpad_med_wide = ON_OFF_graphpad_med_wide.apply(lambda x: sorted(x, key=pd.isnull), 0).dropna(how='all')
ON_OFF_graphpad_med_wide.to_csv(savepath_data / 'T4_PSI_ON_OFF_edges.csv', sep=',', float_format='%.5f')
ON_OFF_graphpad_lob_wide = ON_OFF_graphpad_lob.pivot(index='fly', columns='order_genotype', values='ON_OFF_selectivity').reset_index().drop(columns='fly')
ON_OFF_graphpad_lob_wide = ON_OFF_graphpad_lob_wide.apply(lambda x: sorted(x, key=pd.isnull), 0).dropna(how='all')
ON_OFF_graphpad_lob_wide.to_csv(savepath_data / 'T5_PSI_ON_OFF_edges.csv', sep=',', float_format='%.5f')

In [ ]:
pd.DataFrame(ON_OFF_graphpad.groupby(['order_genotype', 'roi_type']).ON_OFF_selectivity.mean()).reset_index()

plotting_parameters = [
    {'genotype': 'T4T5ad', 'axis': 0},
    {'genotype': 'HisCl1_RNAi', 'axis': 1},
    {'genotype': 'Ort_RNAi', 'axis': 3},
    {'genotype': 'Rdl_RNAi', 'axis': 2},
    {'genotype': 'Rdl_Ort_RNAi', 'axis': 4},
]

fig, axd = plt.subplot_mosaic([['A1', 'A2', 'B1', 'B2', 'C1', 'C2', 'D1', 'D2', 'E1', 'E2'],
                               ['I1', 'I2', 'J1', 'J2', 'K1', 'K2', 'L1', 'L2', 'M1', 'M2'],
                               ['AI', 'AI', 'BJ', 'BJ', 'CK', 'CK', 'DL', 'DL', 'EM', 'EM'],
                               ['A', 'A', 'B', 'B', 'C', 'C', 'D', 'D', 'E', 'E']],
    figsize=(12 * cm_inch, 8 * cm_inch),
    dpi=300,
    gridspec_kw={
        "wspace": 0.2,
        "hspace": 0.3,
        "height_ratios": [1, 1, 1, 1],
    },
                              
)

axis_alphabet = [letter for letter in 'ABCDEFGHIJKLMNOP']

for plot_type, plot_data in [('responses_T4', plot_ON_OFF.query("roi_type == 'med'")), ('responses_T5', plot_ON_OFF.query("roi_type == 'lob'")), ('peaks', plot_ON_OFF_quantify), ('selectivity', ON_OFF_graphpad)]:
    for subgroup in plotting_parameters:
        genotype = subgroup['genotype']
        subgroup_data = plot_data.query("genotype == @genotype")

        if plot_type == 'responses_T4':
            axis_letter = axis_alphabet[subgroup['axis']]
            n_flies = len(subgroup_data.fly.unique())                        
        
        if plot_type == 'responses_T5':
            axis_letter = axis_alphabet[subgroup['axis'] + 8]
            n_flies = len(subgroup_data.fly.unique())

        if (plot_type == 'responses_T4') or (plot_type == 'responses_T5'):
                          
            # ON responses    
            sns.lineplot(data=subgroup_data.query("polarity == 1"), x='relative_time', y='df/f_averaged', ax=axd[axis_letter + '1'], color=palette_genotypes[genotype], estimator='mean', linewidth = 1, errorbar='sd', err_kws={'linewidth': 0})
            axd[axis_letter + '1'].text(-0.4, 4.3, genotype_formatting[genotype] + '\nn = ' + str(n_flies), color=palette_genotypes[genotype], fontsize=6, zorder=100)
            # OFF responses    
            sns.lineplot(data=subgroup_data.query("polarity == -1"), x='relative_time', y='df/f_averaged', ax=axd[axis_letter + '2'], color=palette_genotypes[genotype], estimator='mean', linewidth = 1, errorbar='sd', err_kws={'linewidth': 0})
            axd[axis_letter + '2'].fill_between([-0.8, 1.5], -0.5, 4.1, color='k', alpha=0.2, lw=0)
            
            for axis_to_change in [axis_letter + '1', axis_letter + '2']:
                axd[axis_to_change].set_ylim([-0.5, 4.1])
                axd[axis_to_change].set_xlim([-0.8, 1.5])  
                axd[axis_to_change].set_yticks([0, 1])
                axd[axis_to_change].set_ylabel('ΔF/F')
                axd[axis_to_change].set_xlabel('Time (s)')
                axd[axis_to_change].spines[['left', 'bottom', 'right', 'top']].set_visible(False) 
                axd[axis_to_change].yaxis.set_visible(False) 
                axd[axis_to_change].xaxis.set_visible(False) 
                if axis_to_change == 'A1':
                    axd[axis_to_change].spines[['left']].set_visible(True) 
                    axd[axis_to_change].yaxis.set_visible(True) 
                if axis_to_change == 'I1':
                    axd[axis_to_change].spines[['left', 'bottom']].set_visible(True) 
                    axd[axis_to_change].yaxis.set_visible(True) 
                    axd[axis_to_change].xaxis.set_visible(True) 
        
                  
        if plot_type == 'peaks':
            axis_letter = axis_alphabet[subgroup['axis']] + axis_alphabet[subgroup['axis'] + 8]
            n_flies = 0
            
            sns.barplot(x="roi_type", y="df/f_averaged", hue="polarity", order=['med', 'lob'], #palette=palette_genotypes, 
                          data=subgroup_data, ax=axd[axis_letter], legend=None, estimator='mean', errorbar = 'se', err_kws={'linewidth': 1, 'c': 'k'}, linewidth=0.5, width=0.8)
            
            count = 0
            for patch in axd[axis_letter].patches:
                if count == 0 or count == 1:
                    patch.set_facecolor(mpl.colors.to_rgba(palette_genotypes[genotype], 0.2))
                    patch.set_edgecolor(mpl.colors.to_rgba(palette_genotypes[genotype], 1))
                elif count == 2 or count == 3:
                    patch.set_facecolor(mpl.colors.to_rgba(palette_genotypes[genotype], 0.6))
                    patch.set_edgecolor(mpl.colors.to_rgba(palette_genotypes[genotype], 1))
                count = count + 1
           
            axd[axis_letter].set_ylabel('Response ΔF/F')
            axd[axis_letter].set_xlabel('')
            axd[axis_letter].set_ylim([0, 6])
            axd[axis_letter].set_yticks([0, 2, 4, 6])
            axd[axis_letter].margins(x=0.25)
            axd[axis_letter].spines[['top', 'right']].set_visible(False) 
            if axis_letter != 'AI':
                axd[axis_letter].spines[['left']].set_visible(False) 
                axd[axis_letter].yaxis.set_visible(False) 

            subgroup_data_plot = subgroup_data.copy()
            subgroup_data_plot['plot_x'] = subgroup_data_plot.apply(lambda row: 0 if row.roi_type == 'med' else 1, axis=1)
            subgroup_data_plot['plot_x'] = subgroup_data_plot.apply(lambda row: row.plot_x - 0.2 if row.polarity == 'ON' else row.plot_x + 0.2, axis=1)

            sns.lineplot(data=subgroup_data_plot, x='plot_x', y='df/f_averaged', hue='genotype', palette=palette_genotypes, legend=False, alpha=0.3, ax=axd[axis_letter],  units='fly', estimator=None, linewidth=0.5, zorder=-1)

        if plot_type == 'selectivity':
            axis_letter = axis_alphabet[subgroup['axis']]
            n_flies = 0

            sns.barplot(x="roi_type", y="ON_OFF_selectivity", hue="genotype", order=['med', 'lob'], palette=palette_genotypes, 
                          data=subgroup_data, ax=axd[axis_letter], legend=None, estimator='mean', errorbar = 'se', err_kws={'linewidth': 1, 'c': 'k'}, linewidth=0.5, width=0.5)
            dotsize = 2.5
            sns.stripplot(x="roi_type", y="ON_OFF_selectivity", hue="genotype", palette=palette_genotypes, 
                  data=subgroup_data, dodge=True, alpha=0.6, size=dotsize, edgecolor='white', linewidth=0.2, ax=axd[axis_letter], legend=None, zorder=0)

            
            for patch in axd[axis_letter].patches:
                patch.set_facecolor(mpl.colors.to_rgba(palette_genotypes[genotype], 0.4))
                patch.set_edgecolor(mpl.colors.to_rgba(palette_genotypes[genotype], 1))
                         
            axd[axis_letter].set_ylim([-1, 1])
            axd[axis_letter].set_yticks([-1, 0, 1])
            axd[axis_letter].set_xticks(['med', 'lob'])
            axd[axis_letter].set_xticklabels(['T4', 'T5'])
            axd[axis_letter].spines[['top', 'right']].set_visible(False)
            axd[axis_letter].set_xlabel('')
            axd[axis_letter].margins(x=0.4)
            axd[axis_letter].set_ylabel('PSI')
            if axis_letter != 'A':
                axd[axis_letter].spines[['left']].set_visible(False) 
                axd[axis_letter].yaxis.set_visible(False)  
            axd[axis_letter].axhline(c='k', zorder=0, lw=0.5)


        for i in ['A1', 'A2', 'B1', 'B2', 'C1', 'C2', 'D1', 'D2', 'E1', 'E2', 'I1', 'I2', 'J1', 'J2', 'K1', 'K2', 'L1', 'L2', 'M1', 'M2', 'AI', 'AI', 'BJ', 'BJ', 'CK', 'CK', 'DL', 'DL', 'EM', 'EM', 'A', 'B', 'C', 'D', 'E']:
            for item in ([axd[i].xaxis.label, axd[i].yaxis.label] + axd[i].get_xticklabels() + axd[i].get_yticklabels()):
                item.set_fontsize(6)
            
figtitle = "ON_OFF_selectivity_traces_mean_sd_PSI.pdf"
plt.savefig(savepath_fig / figtitle)